In [1]:
from dotenv import load_dotenv
import os

import diqrng

load_dotenv()

True

In [2]:
def save_bits(filename: str, bitstring: str):
    bytestrings = [bitstring[i:i+8] for i in range(0, len(bitstring), 8)]
    ba = bytearray(int(byte, 2) for byte in bytestrings)
    with open(f"output/{filename}.bin", "wb") as file:
        file.write(bytes(ba))

# Demos

## Simulated Backend

To run on a simulated backend, pass `None` as the `token`.

Generate random numbers across two rounds and check CHSH violations across the other two.

In [ ]:
try:
    diqrng.generate([0, 1, 0, 1, 0, 1, 0, 1], 4, token=None, bases=diqrng.BASES)
except diqrng.AbortGeneration as e:
    print(e)

Due to noise, the CHSH inequality will not be violated. Repeat with increased uncertainty and increase the number of shots in check rounds to reduce variance.

In [ ]:
uncertainty = 0.3
print("Producing random numbers with minimum CHSH violation:", diqrng.get_threshold(uncertainty))
bs = diqrng.generate([0, 1, 0, 1, 0, 1, 0, 1], 4, token=None, bases=diqrng.BASES, uncertainty=uncertainty, check_shots=10000)
save_bits("16_FakeKyiv_sim", bs)

## IBMQ Backend

To run on an IBMQ computer, pass your IMBQ API token to `token`. The script below assumes you have saved your token under `IMBQ_API_TOKEN` in a local `.env` file.

Note that generation rounds are placed next to one another for efficiecy purposes.

In [ ]:
uncertainty = 0.3
print("Producing random numbers with minimum CHSH violation:", diqrng.get_threshold(uncertainty))
bs = diqrng.generate([0, 0, 0, 0, 1, 1, 1, 1], 4, token=os.getenv("IBMQ_API_TOKEN"), bases=diqrng.BASES, uncertainty=uncertainty, check_shots=10000, backend="ibm_brisbane")
save_bits("16_Brisbane_ibm", bs)

# Investigation

In [3]:
import numpy as np
import plotly.express as px
import pandas as pd
import time
import math

## Time Efficiency

### Impact of parameters on the time taken to generate qubits.

In [ ]:
range_of_pairs = range(1, 10)
range_of_shots = [1, 10, 50, 100, 500, 1000, 5000, 10000, 50000, 100000]
time_taken = {"pairs":[], "shots":[], "time":[]}


for num_pairs in range_of_pairs:
    circuit = diqrng.ChshCircuit(num_pairs=num_pairs, token=None)
    print(f"Number of pairs: {num_pairs}")
    if num_pairs > 7:
        range_of_shots.pop()
    for num_shots in range_of_shots:
        start = time.time()
        circuit.generate_numbers(num_shots=num_shots)
        end = time.time()
        print(f"Time taken for {num_shots} number of shots: {end - start}")
        time_taken["pairs"].append(num_pairs)
        time_taken["shots"].append(num_shots)
        time_taken["time"].append(end - start)

time_taken = pd.DataFrame(time_taken)

plot = px.line(time_taken, x="shots", y="time", color="pairs", title="Time taken to generate random numbers with different number of pairs and shots")
plot.write_image("output/time_generate.png")
plot.show()

### Impact of parameters on the time taken to measure CHSH observables

In [ ]:
range_of_pairs = range(1, 10)
range_of_shots = [1, 10, 50, 100, 500, 1000, 5000, 10000, 50000, 100000]
time_taken = {"pairs":[], "shots":[], "time":[]}


for num_pairs in range_of_pairs:
    circuit = diqrng.ChshCircuit(num_pairs=num_pairs, token=None)
    print(f"Number of pairs: {num_pairs}")
    if num_pairs > 6:
        range_of_shots.pop()
    for num_shots in range_of_shots:
        start = time.time()
        for basis in diqrng.BASES:
            circuit.measure_chsh_basis(basis, num_shots=num_shots)
        end = time.time()
        print(f"Time taken for {num_shots} number of shots: {end - start}")
        time_taken["pairs"].append(num_pairs)
        time_taken["shots"].append(num_shots)
        time_taken["time"].append(end - start)

time_taken = pd.DataFrame(time_taken)

plot = px.line(time_taken, x="shots", y="time", color="pairs", title="Time taken to measure CHSH observables with different number of pairs and shots")
plot.write_image("output/time_measure.png")
plot.show()

## Generating 10M qubits

In [4]:
num_pairs = 63
num_gen_shots = 1588
num_check_shots = 10000
num_gen_rounds = 100
num_check_rounds = 12
uncertainty = 2 * math.sqrt(2) / (2 * math.sqrt(2) - 2)

random_indexes = np.random.choice(range(num_gen_rounds + num_check_rounds), num_check_rounds, replace=False)
rounds = [0 for _ in range(num_gen_rounds + num_check_rounds)]
for index in random_indexes:
    rounds[index] = 1
bases = [diqrng.BASES[0] for _ in range(num_check_rounds)]
for basis in diqrng.BASES[1:] * 3:
    random_index = np.random.choice(range(num_check_rounds), 1)[0]
    while bases[random_index] != diqrng.BASES[0]:
        random_index = (random_index + 1) % num_check_rounds
    bases[random_index] = basis

In [5]:
print("Producing random numbers with minimum CHSH violation:", diqrng.get_threshold(uncertainty))

generated_numbers = diqrng.generate(
    rounds,
    num_pairs,
    token=os.getenv("IBMQ_API_TOKEN"),
    bases=bases,
    gen_shots=num_gen_shots,
    check_shots=num_check_shots,
    uncertainty=uncertainty,
    backend="ibm_brisbane"
)

save_bits("10M_Brisbane", generated_numbers)

Producing random numbers with minimum CHSH violation: 0.0
Violations
[2.3606, 2.3621333333333334, 2.542866666666667, 2.3996, 2.6330666666666662, 2.5730666666666666, 2.5288666666666666, 2.3384666666666667, 2.3478666666666665, 2.498733333333333, 2.5380666666666665, 2.567466666666667, 2.3132, 2.7012, 2.0862666666666665, 2.5648666666666666, 2.5207333333333333, 2.3622666666666667, 2.5238000000000005, 2.2058666666666666, 2.4763333333333333, 2.1331999999999995, 2.517666666666667, 2.5323333333333338, 2.4758666666666667, 2.434066666666667, 2.2465333333333333, 2.5070666666666663, 2.4872666666666667, 2.3098, 2.264866666666667, 2.5808, 2.555266666666667, 2.006333333333333, 2.4481333333333333, 2.5080666666666667, 2.5083333333333333, 2.2909333333333333, 2.4439333333333333, 2.5172666666666665, 2.426, 2.613, 2.4239333333333333, 2.5488666666666666, 2.4087333333333336, 2.0113333333333334, 2.5933333333333333, 2.5114666666666663, 2.4702, 2.3726000000000003, 2.4671999999999996, 2.3823333333333334, 2.6648, 

In [6]:
violations = [2.3606, 2.3621333333333334, 2.542866666666667, 2.3996, 2.6330666666666662, 2.5730666666666666, 2.5288666666666666, 2.3384666666666667, 2.3478666666666665, 2.498733333333333, 2.5380666666666665, 2.567466666666667, 2.3132, 2.7012, 2.0862666666666665, 2.5648666666666666, 2.5207333333333333, 2.3622666666666667, 2.5238000000000005, 2.2058666666666666, 2.4763333333333333, 2.1331999999999995, 2.517666666666667, 2.5323333333333338, 2.4758666666666667, 2.434066666666667, 2.2465333333333333, 2.5070666666666663, 2.4872666666666667, 2.3098, 2.264866666666667, 2.5808, 2.555266666666667, 2.006333333333333, 2.4481333333333333, 2.5080666666666667, 2.5083333333333333, 2.2909333333333333, 2.4439333333333333, 2.5172666666666665, 2.426, 2.613, 2.4239333333333333, 2.5488666666666666, 2.4087333333333336, 2.0113333333333334, 2.5933333333333333, 2.5114666666666663, 2.4702, 2.3726000000000003, 2.4671999999999996, 2.3823333333333334, 2.6648, 2.4437333333333333, 2.6231333333333335, 1.8612, 2.5622, 2.4141333333333335, 2.1438, 2.598333333333333, 2.1824666666666666, 2.1654, 2.3162666666666665]

for threshold in [i / 10 for i in range(20, 30)]:
    print(f"Number of violations below {threshold}:", len([v for v in violations if v < threshold]))

Number of violations below 2.0: 1
Number of violations below 2.1: 4
Number of violations below 2.2: 8
Number of violations below 2.3: 12
Number of violations below 2.4: 23
Number of violations below 2.5: 37
Number of violations below 2.6: 58
Number of violations below 2.7: 62
Number of violations below 2.8: 63
Number of violations below 2.9: 63
